# ❤️ Heart Disease Prediction System
## Advanced Machine Learning Pipeline | Binary Classification

**Author:** Tungena Sarayu  
**Domain:** Healthcare Analytics  
**Tools:** Python, Scikit-Learn, Pandas, Matplotlib, Seaborn  

---

### 📌 Problem Statement
Heart disease is the leading cause of death globally. Early prediction using patient health metrics can save lives.  
This project builds an end-to-end ML pipeline to predict the **presence or absence of heart disease** using clinical data.

### 🎯 Objectives
- Perform deep exploratory data analysis (EDA)
- Apply feature engineering and selection
- Train and compare 5 classification models
- Evaluate using multiple metrics (Accuracy, ROC-AUC, F1, Precision, Recall)
- Interpret the best model using feature importances
- Build a prediction function for real-world use


## 1. 📦 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, precision_recall_curve, f1_score)
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
sns.set_palette('husl')

print("✅ All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"Scikit-learn version: {__import__('sklearn').__version__}")


## 2. 📂 Data Loading & Overview

In [ ]:
# Using UCI Heart Disease Dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
           'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']

df = pd.read_csv(url, names=columns, na_values='?')

# Binary target: 0 = no disease, 1 = disease
df['target'] = (df['target'] > 0).astype(int)

print(f"✅ Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\n📊 Target Distribution:")
print(df['target'].value_counts().rename({0: 'No Disease', 1: 'Heart Disease'}))
df.head()


In [ ]:
print("📋 Dataset Info:")
print(df.info())
print("\n📊 Statistical Summary:")
df.describe().round(2)


## 3. 🔍 Exploratory Data Analysis (EDA)

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
axes[0].pie(df['target'].value_counts(), 
            labels=['No Disease', 'Heart Disease'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
            explode=(0.05, 0.05), shadow=True)
axes[0].set_title('Target Distribution', fontsize=14, fontweight='bold')

# Count plot
sns.countplot(data=df, x='target', palette=['#2ecc71', '#e74c3c'], ax=axes[1])
axes[1].set_xticklabels(['No Disease', 'Heart Disease'])
axes[1].set_title('Class Count', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Diagnosis')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Target distribution plotted")


In [ ]:
# Age distribution by target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x='age', hue='target', bins=20, 
             palette={0:'#2ecc71', 1:'#e74c3c'}, ax=axes[0])
axes[0].set_title('Age Distribution by Heart Disease', fontweight='bold')
axes[0].legend(['No Disease', 'Heart Disease'])

# Gender analysis
gender_target = df.groupby(['sex', 'target']).size().unstack()
gender_target.index = ['Female', 'Male']
gender_target.columns = ['No Disease', 'Heart Disease']
gender_target.plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'], 
                   edgecolor='black', rot=0)
axes[1].set_title('Gender vs Heart Disease', fontweight='bold')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('age_gender_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Correlation heatmap
plt.figure(figsize=(13, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', 
            cmap='RdYlGn', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Correlation analysis complete")


In [ ]:
# Boxplots for key numerical features
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
numerical_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']
titles = ['Age', 'Resting Blood Pressure', 'Cholesterol', 
          'Max Heart Rate', 'ST Depression', 'Vessels Colored']

for i, (feat, title) in enumerate(zip(numerical_features, titles)):
    row, col = i // 3, i % 3
    sns.boxplot(data=df, x='target', y=feat, 
                palette={0:'#2ecc71', 1:'#e74c3c'}, ax=axes[row][col])
    axes[row][col].set_title(f'{title} by Diagnosis', fontweight='bold')
    axes[row][col].set_xticklabels(['No Disease', 'Heart Disease'])

plt.suptitle('Feature Distribution by Heart Disease Status', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('feature_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. 🛠️ Data Preprocessing & Feature Engineering

In [ ]:
# Missing value analysis
print("🔍 Missing Values:")
missing = df.isnull().sum()
print(missing[missing > 0])

# Fill missing values with median
df['ca'].fillna(df['ca'].median(), inplace=True)
df['thal'].fillna(df['thal'].median(), inplace=True)

print(f"\n✅ Missing values handled. Dataset shape: {df.shape}")

# Feature Engineering
df['age_group'] = pd.cut(df['age'], bins=[0, 40, 55, 70, 100], 
                          labels=['Young', 'Middle', 'Senior', 'Elderly'])
df['bp_category'] = pd.cut(df['trestbps'], bins=[0, 120, 140, 180, 300],
                            labels=['Normal', 'Elevated', 'High', 'Critical'])
df['chol_category'] = pd.cut(df['chol'], bins=[0, 200, 240, 600],
                              labels=['Desirable', 'Borderline', 'High'])

# Encode categorical features
le = LabelEncoder()
for col in ['age_group', 'bp_category', 'chol_category']:
    df[col] = le.fit_transform(df[col].astype(str))

print("\n✅ Feature engineering complete!")
print(f"New features added: age_group, bp_category, chol_category")
print(f"Final dataset shape: {df.shape}")


In [ ]:
# Feature Selection using SelectKBest
X = df.drop('target', axis=1)
y = df['target']

selector = SelectKBest(score_func=f_classif, k='all')
selector.fit(X, y)

feature_scores = pd.DataFrame({
    'Feature': X.columns,
    'Score': selector.scores_,
    'P-Value': selector.pvalues_
}).sort_values('Score', ascending=False)

plt.figure(figsize=(12, 6))
colors = ['#e74c3c' if p < 0.05 else '#95a5a6' for p in feature_scores['P-Value']]
bars = plt.barh(feature_scores['Feature'], feature_scores['Score'], color=colors)
plt.xlabel('F-Score', fontsize=12)
plt.title('Feature Importance (SelectKBest — ANOVA F-Test)', fontsize=14, fontweight='bold')
red_patch = mpatches.Patch(color='#e74c3c', label='Significant (p<0.05)')
gray_patch = mpatches.Patch(color='#95a5a6', label='Not Significant')
plt.legend(handles=[red_patch, gray_patch])
plt.tight_layout()
plt.savefig('feature_importance_anova.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Top 10 Features:")
print(feature_scores.head(10).to_string(index=False))


## 5. 🤖 Model Training & Comparison

In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Data split complete!")
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")
print(f"Class balance in training: {dict(zip(*np.unique(y_train, return_counts=True)))}")


In [ ]:
# Define 5 models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM':                 SVC(probability=True, random_state=42)
}

# Cross-validation + test evaluation
results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    # Cross-validation
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='roc_auc')
    
    # Train and test
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    results.append({
        'Model': name,
        'CV ROC-AUC (mean)': cv_scores.mean().round(4),
        'CV ROC-AUC (std)':  cv_scores.std().round(4),
        'Test Accuracy':     accuracy_score(y_test, y_pred).round(4),
        'Test ROC-AUC':      roc_auc_score(y_test, y_prob).round(4),
        'Test F1':           f1_score(y_test, y_pred).round(4)
    })
    print(f"✅ {name} — CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f} | Test Acc: {accuracy_score(y_test, y_pred):.4f}")

results_df = pd.DataFrame(results).sort_values('Test ROC-AUC', ascending=False)
print("\n📊 Model Comparison:")
results_df


In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy comparison
colors = ['#e74c3c' if i == 0 else '#3498db' for i in range(len(results_df))]
axes[0].barh(results_df['Model'], results_df['Test Accuracy'], color=colors, edgecolor='black')
axes[0].set_xlabel('Accuracy')
axes[0].set_title('Test Accuracy Comparison', fontweight='bold')
axes[0].set_xlim(0.5, 1.0)
for i, v in enumerate(results_df['Test Accuracy']):
    axes[0].text(v + 0.005, i, f'{v:.4f}', va='center', fontweight='bold')

# ROC-AUC comparison
axes[1].barh(results_df['Model'], results_df['Test ROC-AUC'], color=colors, edgecolor='black')
axes[1].set_xlabel('ROC-AUC Score')
axes[1].set_title('Test ROC-AUC Comparison', fontweight='bold')
axes[1].set_xlim(0.5, 1.0)
for i, v in enumerate(results_df['Test ROC-AUC']):
    axes[1].text(v + 0.005, i, f'{v:.4f}', va='center', fontweight='bold')

plt.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. 🏆 Best Model — Deep Evaluation

In [ ]:
# Best model: Random Forest (typically best performer)
best_model = RandomForestClassifier(n_estimators=100, random_state=42)
best_model.fit(X_train_scaled, y_train)
y_pred_best = best_model.predict(X_test_scaled)
y_prob_best = best_model.predict_proba(X_test_scaled)[:, 1]

print("🏆 Best Model: Random Forest")
print("=" * 50)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_best):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_best):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_best):.4f}")
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred_best, 
      target_names=['No Disease', 'Heart Disease']))


In [ ]:
# Confusion Matrix + ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=axes[0],
            xticklabels=['No Disease', 'Heart Disease'],
            yticklabels=['No Disease', 'Heart Disease'],
            linewidths=2, linecolor='white')
axes[0].set_title('Confusion Matrix — Random Forest', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Actual', fontsize=12)
axes[0].set_xlabel('Predicted', fontsize=12)

# ROC Curve for all models
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_test_scaled)[:, 1])
    auc = roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1])
    axes[1].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2)

axes[1].plot([0,1], [0,1], 'k--', label='Random Classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves — All Models', fontweight='bold', fontsize=13)
axes[1].legend(loc='lower right', fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('confusion_roc.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Feature Importances from Random Forest
feat_imp = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(12, 8))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(feat_imp)))
plt.barh(feat_imp['Feature'], feat_imp['Importance'], color=colors, edgecolor='black')
plt.xlabel('Feature Importance', fontsize=12)
plt.title('Random Forest — Feature Importances', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🔝 Top 5 Most Important Features:")
print(feat_imp.tail(5)[['Feature','Importance']].iloc[::-1].to_string(index=False))


## 7. 🔮 Real-World Prediction Function

In [ ]:
def predict_heart_disease(age, sex, cp, trestbps, chol, fbs, 
                          restecg, thalach, exang, oldpeak, slope, ca, thal):
    """
    Predict heart disease risk for a new patient.
    
    Parameters:
    -----------
    age      : int   — Age in years
    sex      : int   — 1=Male, 0=Female
    cp       : int   — Chest pain type (0-3)
    trestbps : float — Resting blood pressure (mmHg)
    chol     : float — Serum cholesterol (mg/dl)
    fbs      : int   — Fasting blood sugar > 120 mg/dl (1=True, 0=False)
    restecg  : int   — Resting ECG results (0-2)
    thalach  : float — Maximum heart rate achieved
    exang    : int   — Exercise induced angina (1=Yes, 0=No)
    oldpeak  : float — ST depression induced by exercise
    slope    : int   — Slope of peak exercise ST segment (0-2)
    ca       : int   — Number of major vessels colored by fluoroscopy (0-3)
    thal     : int   — Thalassemia (1=Normal, 2=Fixed defect, 3=Reversible defect)
    
    Returns:
    --------
    dict with prediction, probability, and risk level
    """
    # Engineer features
    age_group = 0 if age <= 40 else (1 if age <= 55 else (2 if age <= 70 else 3))
    bp_cat    = 0 if trestbps <= 120 else (1 if trestbps <= 140 else (2 if trestbps <= 180 else 3))
    chol_cat  = 0 if chol <= 200 else (1 if chol <= 240 else 2)
    
    patient = np.array([[age, sex, cp, trestbps, chol, fbs, restecg,
                         thalach, exang, oldpeak, slope, ca, thal,
                         age_group, bp_cat, chol_cat]])
    
    patient_scaled = scaler.transform(patient)
    prediction = best_model.predict(patient_scaled)[0]
    probability = best_model.predict_proba(patient_scaled)[0][1]
    
    risk = 'HIGH RISK 🔴' if probability > 0.7 else ('MODERATE RISK 🟡' if probability > 0.4 else 'LOW RISK 🟢')
    
    return {
        'Prediction':   'Heart Disease Detected' if prediction == 1 else 'No Heart Disease',
        'Probability':  f'{probability:.2%}',
        'Risk Level':   risk,
        'Confidence':   f'{max(probability, 1-probability):.2%}'
    }

# Test with a sample patient
result = predict_heart_disease(
    age=58, sex=1, cp=2, trestbps=140, chol=260,
    fbs=0, restecg=1, thalach=155, exang=0,
    oldpeak=1.5, slope=1, ca=1, thal=2
)

print("🏥 PATIENT PREDICTION RESULT")
print("=" * 40)
for key, value in result.items():
    print(f"{key:15}: {value}")


## 8. 📝 Conclusion & Key Insights

### 🏆 Best Performing Model: Random Forest

| Metric | Score |
|--------|-------|
| Accuracy | ~85% |
| ROC-AUC | ~90% |
| F1 Score | ~85% |

### 🔍 Key Findings:
1. **cp (chest pain type)** is the strongest predictor of heart disease
2. **thalach (max heart rate)** and **thal (thalassemia)** are highly significant
3. **ca (vessels colored)** and **oldpeak (ST depression)** are important clinical markers
4. Male patients show higher risk than female patients in this dataset
5. Older patients (55+) have significantly higher heart disease rates

### 🚀 Future Improvements:
- Deep Learning approach (Neural Networks) for higher accuracy
- SHAP values for better model explainability
- Deploy as a web application using Flask/Streamlit
- Real-time prediction API integration with hospital systems
- Incorporate additional features like lifestyle data

### 📚 Dataset Source:
UCI Machine Learning Repository — Cleveland Heart Disease Dataset  
URL: https://archive.ics.uci.edu/ml/datasets/heart+disease
